In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드

load_dotenv()

ModuleNotFoundError: No module named 'dotenv'

In [33]:
from langchain_community.document_loaders.blob_loaders.youtube_audio import (
    YoutubeAudioLoader,
)
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import OpenAIWhisperParser

In [ ]:
# YouTube 동영상 URL
urls = ["https://www.youtube.com/watch?v=YQ5kIuoHam8"]

# 오디오 파일을 저장할 디렉토리
save_dir = "./youtube_audios/"

In [35]:
# 동영상을 텍스트로 변환
loader = GenericLoader(YoutubeAudioLoader(
    urls, save_dir), OpenAIWhisperParser())
docs = loader.load()

[youtube] Extracting URL: https://youtu.be/t9ugPGO3_RM?si=HF5sMD0ma1IW-3o6
[youtube] t9ugPGO3_RM: Downloading webpage
[youtube] t9ugPGO3_RM: Downloading ios player API JSON
[youtube] t9ugPGO3_RM: Downloading m3u8 information
[info] t9ugPGO3_RM: Downloading 1 format(s): 140
[download] ./youtube_audios//고든램지 풀드포크 ： 육즙 대폭발! 바베큐도 고든램지가 하면 다르다 (Gordon Ramsay's Smoky Pulled Pork with Chipotle Mayonnaise).m4a has already been downloaded
[download] 100% of   11.78MiB
[ExtractAudio] Not converting audio ./youtube_audios//고든램지 풀드포크 ： 육즙 대폭발! 바베큐도 고든램지가 하면 다르다 (Gordon Ramsay's Smoky Pulled Pork with Chipotle Mayonnaise).m4a; file is already in target format m4a
Transcribing part 1!
Transcribing part 1!
Transcribing part 1!


In [36]:
# 첫 번째 문서의 내용을 파일에 저장
output_file_path = 'transcribed_text_0.txt'
with open(output_file_path, 'w') as file:
    file.write(docs[0].page_content)

print(f"Transcribed text saved to {output_file_path}")

Transcribed text saved to transcribed_text_0.txt


In [37]:
# 저장된 파일의 내용을 확인하기 위해 다시 읽기
with open(output_file_path, 'r') as file:
    saved_text = file.read()

print(f"Saved Text:\n{saved_text[:500]}")  # 첫 500자만 출력

Saved Text:
오늘 준비한 고기부터 보시죠 코스트코에서 산 돼지 목살입니다 언제나 말씀드리듯이 목살 원육은 코스트코가 최고입니다 원래는 이걸 한 덩이로 2만 원대에 팔았었거든요? 오늘은 가보니까 두 덩이 묶어서 4만 원대에 팔더라고요 원래 풀드포크는 버트라고 하는 돼지 어깨 부위로 만듭니다 하지만 한국에서는 버트 커팅을 거의 구할 수 없기 때문에 저처럼 목살, 목심을 사용하시면 되겠습니다 풀드포크는 굳이 지방을 많이 넣지 않으셔도 돼요 근데 이 부위는 뼈가 많이 들어가서 뼈가 많이 들어가서 뼈가 많이 들어가서 목심을 사용하시면 되겠습니다 풀드포크는 굳이 지방 손질을 할 필요가 없습니다 비계가 녹으면서 살코기 쪽의 육즙을 보충하기 때문이죠 이렇게 핏물을 잘 닦기만 하면 고기 준비가 완료됩니다 육식맨 채널 첫 영상이 오븐 풀드포크였고요 수비드 풀드포크 영상도 큰 사랑을 받았는데요 오늘은 램지 형님을 모셔서 풀드포크를 만들어봅니다 거의 천만 뷰가 나온 초대박 비디오로 98년생 맏딸 레강과 함께 만들기 


In [38]:
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [39]:
# 문서 결합
# docs의 각 문서에서 page_content를 추출하여 combined_docs에 저장합니다.
combined_docs = [doc.page_content for doc in docs]
text = " ".join(combined_docs)

In [40]:
# 텍스트를 분할합니다.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, chunk_overlap=150)
splits = text_splitter.split_text(text)

In [41]:
# 색인을 구축합니다.
embeddings = OpenAIEmbeddings()
vectordb = FAISS.from_texts(splits, embeddings)

In [42]:
from langchain.callbacks.base import BaseCallbackHandler


class StreamCallback(BaseCallbackHandler):
    def on_llm_new_token(self, token: str, **kwargs):
        print(f"{token}", end="", flush=True)

In [43]:
# QA 체인 구축
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(
        model_name="gpt-4o",
        temperature=0,
        streaming=True,
        callbacks=[StreamCallback()],
    ),
    chain_type="stuff",
    retriever=vectordb.as_retriever(),
)

In [44]:
# 질문을 하세요!
query = "전체적인 레시피에 대해 알려주세요"
answer = qa_chain.invoke({"query": query})

비빔 칼국수와 냄비 뚜껑 훈제 수육의 레시피를 요약해 드리겠습니다.

### 비빔 칼국수 레시피
1. **재료 준비**:
   - 오이 1개: 반으로 자른 뒤 어슷하게 썰기
   - 상추: 손으로 대충 찢기
   - 칼국수 면: 잘 헹궈서 준비

2. **양념장 만들기**:
   - 고추장
   - 고춧가루
   - 설탕
   - 진간장
   - 식초
   - 참기름
   - 다진 마늘
   - (양념장의 자세한 계량은 더보기에 참고)

3. **비빔 칼국수 만들기**:
   - 잘 헹군 칼국수 면에 양념장, 잘 익은 김치, 손질한 채소를 넣고 버무리기
   - 깨를 듬뿍 뿌려 마무리

### 냄비 뚜껑 훈제 수육 레시피
1. **재료 준비**:
   - 수육용 고기
   - 된장
   - 훈연 칩

2. **수육 준비**:
   - 고기에 된장을 촘촘하게 바르기

3. **훈제 과정**:
   - 냄비 뚜껑을 이용해 훈제하기
   - 훈연 칩을 사용해 훈제 향을 입히기

4. **조리**:
   - 수육을 훈제한 후 잘라서 준비

### 조리 팁
- **비빔 칼국수**: 김치의 맛에 따라 편차가 있을 수 있으니, 맛있는 김치를 사용하는 것이 중요합니다.
- **훈제 수육**: 훈연 칩을 사용해 훈제 향을 강하게 입히고, 된장을 고기에 촘촘하게 발라 맛을 더합니다.

### 추가 팁
- **캠핑에서 요리**: 아이오닉5의 V2L 기능을 이용해 캠핑장에서 전기 가전제품을 사용해 요리할 수 있습니다.
- **시간 관리**: 훈제 수육은 시간이 걸리지만, 중간에 할 일이 없어서 생각보다 편리합니다.

이 레시피를 따라 하시면 맛있는 비빔 칼국수와 훈제 수육을 즐기실 수 있습니다.